In [ ]:
import pandas as pd
import sys
import os

if "Desktop" in os.getcwd():
    base_path = "."
else:
    base_path = "."

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import (
    Datasets,
    DATA_FOLDER_PREFIX,
    Dataset,
    NELA_SOURCE_LABELS,
)
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv, save_to_csv, read_model_feature_injected_csv
from src.weak_labeling.consts.globals import NLP

# covid_df: pd.DataFrame = read_raw_data(Datasets.NELA_COVID_2021_DATASET)
# elections_df: pd.DataFrame = read_raw_data(Datasets.NELA_ELECTIONS_DATASET_SAMPLED)

In [ ]:
import json

weak_labeler_runs_path = os.path.join(
    get_project_root(), f"data/with_weak_labels/runs.json"
)
f = open(weak_labeler_runs_path)
old_runs = json.load(f)
f.close()

In [ ]:
run_analysis = {}
for run in old_runs:
    name = run["csv_path"].split('/')[-1].split("-top")[0]
    run_analysis[name] = []

In [ ]:
run_analysis

In [ ]:
for run in old_runs:
    print(run)
    name = run["csv_path"].split('/')[-1].split("-top")[0]
    max_lf = run["lfs"][0][0]
    max_lf_score = run["lfs"][0][1]
    for lf in run["lfs"][1:]:
        if lf[1] > max_lf_score:
            max_lf = lf[0]
            max_lf_score = lf[1]
    current_result = {
        "snorkel_majority_vote_accuracy": run["snorkel_majority_vote"]["accuracy"],
        "snorkel_majority_vote_coverage": run["snorkel_majority_vote"]["coverage"],
        "snorkel_label_model": run["snorkel_label_model"]["accuracy"],
        "hyper_label_model": run["hyper_label_model"]["accuracy"],
        "top_n": run["top_n"],
        "max_lf": max_lf,
        "max_lf_score": max_lf_score,
    }
    run_analysis[name].append(current_result)

In [ ]:
run_analysis

In [ ]:
trained_with_elections_points = []
trained_with_elections_accuracies = []
trained_with_elections_coverages = []

for results in run_analysis["nela-covid-combined-labeled-with-elections-2020"]:
    trained_with_elections_points.append(results["top_n"])
    trained_with_elections_accuracies.append(results["snorkel_majority_vote_accuracy"])
    trained_with_elections_coverages.append(results["snorkel_majority_vote_coverage"])

In [ ]:
trained_with_elections_df = pd.DataFrame(
    {'trained_with_elections_points': trained_with_elections_points,
     'trained_with_elections_accuracies': trained_with_elections_accuracies,
     'trained_with_elections_coverages': trained_with_elections_coverages
    })

In [ ]:
trained_with_elections_df.head(
)

In [ ]:
trained_with_elections_df.to_csv("./src/custom_notebooks/trained_with_elections_df_stats.csv", float_format='%.3f')

In [ ]:
trained_with_elections_accuracies

In [ ]:
import matplotlib.pyplot as plt

plt.plot(trained_with_elections_points, trained_with_elections_accuracies)
plt.plot(trained_with_elections_points, trained_with_elections_coverages, '-.')

plt.xlabel("X-axis data")
plt.ylabel("Y-axis data")
plt.title('multiple plots')
plt.show()

In [ ]:
trained_with_covid_points = []
trained_with_covid_accuracies = []
trained_with_covid_coverages = []

for results in run_analysis["nela-covid-combined-labeled-with-covid-subset"]:
    trained_with_covid_points.append(results["top_n"])
    trained_with_covid_accuracies.append(results["snorkel_majority_vote_accuracy"])
    trained_with_covid_coverages.append(results["snorkel_majority_vote_coverage"])

In [ ]:
for a, b in zip(trained_with_covid_points, trained_with_covid_accuracies):
    print(f"{a}----{b}")

In [ ]:
trained_with_covid_df = pd.DataFrame(
    {'trained_with_covid_points': trained_with_covid_points,
     'trained_with_covid_accuracies': trained_with_covid_accuracies,
     'trained_with_covid_coverages': trained_with_covid_coverages
    })

In [ ]:
import matplotlib.pyplot as plt

plt.plot(trained_with_covid_points, trained_with_covid_accuracies)
plt.plot(trained_with_covid_points, trained_with_covid_coverages, '-.')

plt.xlabel("X-axis data")
plt.ylabel("Y-axis data")
plt.title('multiple plots')
plt.show()

In [ ]:
trained_with_covid_df.head()

In [ ]:
trained_with_covid_df.to_csv("./src/custom_notebooks/trained_with_covid_df_stats.csv", float_format='%.3f')

In [ ]:
highest_results = {}
for key in run_analysis.keys():
    highest_score = -1
    highest_result = None
    for res in run_analysis[key]:
        print(res["snorkel_majority_vote_accuracy"])
        if res["snorkel_majority_vote_accuracy"] > highest_score:
            highest_score = res["snorkel_majority_vote_accuracy"]
            highest_result = res
    highest_results[key] = highest_result

In [ ]:
highest_results

In [ ]:
x = []
y1 = []
y2 = []
for run in old_runs:
    name = run["csv_path"].split('/')[-1].split("-top")[0]
    if name == "nela-covid-combined-labeled-with-covid-subset":
        x.append(run["top_n"])
        y1.append(run["snorkel_majority_vote"]["accuracy"])
        y2.append(run["snorkel_majority_vote"]["coverage"])

In [ ]:
print(x)
print(y1)
print(y2)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(x, y1, label ='accuracy')
plt.plot(x, y2, '-.', label ='coverage')

plt.xlabel("Number of top labeling functions")
plt.ylabel("")
plt.legend()
plt.title('multiple plots')
plt.show()